# Lesson 25: Structure from Motion

This lesson is a capstone: it assembles feature matching (Lesson 18), the essential matrix and pose recovery (Lesson 22), calibration (Lesson 23), and robust estimation (Lesson 24) into a complete pipeline that takes 2D image correspondences from two views and recovers **both** the cameras' relative motion **and** the 3D positions of the points that were being viewed &mdash; **S**tructure **f**rom **M**otion. The one genuinely new ingredient is **triangulation**: turning a matched 2D point pair, plus known camera poses, into a 3D point.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## The pipeline, at a glance

1. **Match features** between two images (Lesson 18: SIFT + ratio test).
2. **Estimate the essential matrix** $E$ from those correspondences, robustly (Lesson 24: RANSAC), using known intrinsics $K$ (Lesson 23: calibration).
3. **Recover relative pose** $(R, t)$ from $E$ (Lesson 22: `cv2.recoverPose`) &mdash; up to an unknown scale on $t$.
4. **Triangulate**: for every matched pair, intersect the two corresponding rays in 3D to recover a 3D point &mdash; new in this lesson.

We build a synthetic two-camera scene (as in Lessons 22-23) so we always have ground truth to check the *entire* pipeline against, end to end.

In [ ]:
rng = np.random.default_rng(0)
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

R1, t1 = np.eye(3), np.zeros(3)
angle = np.radians(15)
R_true = np.array([[np.cos(angle), 0, np.sin(angle)],
                    [0, 1, 0],
                    [-np.sin(angle), 0, np.cos(angle)]])
t_true = np.array([0.5, 0.0, 0.1])

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2_true = K @ np.hstack([R_true, t_true.reshape(3, 1)])

def project(P, points_3d):
    homogeneous = np.hstack([points_3d, np.ones((len(points_3d), 1))])
    projected = (P @ homogeneous.T).T
    return projected[:, :2] / projected[:, 2:3]

points_3d_true = rng.uniform(-1, 1, (50, 3)) + np.array([0, 0, 5])
x1 = project(P1, points_3d_true)
x2 = project(P2_true, points_3d_true)

## Step 1-2: robust essential matrix and pose recovery

Standing in for real (imperfect) feature matches from Lesson 18, we feed the correspondences through the same robust pipeline built in Lessons 22 and 24.

In [ ]:
E, inlier_mask = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)
_, R_estimated, t_estimated, _ = cv2.recoverPose(E, x1, x2, K)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered rotation:\n', np.round(R_estimated, 4))
print('true rotation:\n', np.round(R_true, 4))
print()
print('recovered translation direction:', np.round(t_estimated.ravel(), 4))
print('true translation direction:     ', np.round(t_true / np.linalg.norm(t_true), 4))

As in Lesson 22, `recoverPose` returns a **unit-length** translation direction &mdash; the actual baseline distance between the cameras is fundamentally unrecoverable from image correspondences alone. We'll come back to that.

## Step 3: triangulation

Given two camera projection matrices $P_1, P_2$ (each $3\times4$, mapping a 3D point to a 2D image point in homogeneous coordinates) and a matched pixel pair $(x_1, x_2)$, we want the 3D point $X$ satisfying both $x_1 \propto P_1 X$ and $x_2 \propto P_2 X$. Each view contributes 2 independent linear equations in $X$'s 4 homogeneous unknowns (cross-multiplying out the unknown scale factor), by the same DLT/SVD recipe used for homographies (Lesson 21) and the fundamental matrix (Lesson 22):

$$A = \begin{bmatrix} u_1 P_1^{(3)} - P_1^{(1)} \\ v_1 P_1^{(3)} - P_1^{(2)} \\ u_2 P_2^{(3)} - P_2^{(1)} \\ v_2 P_2^{(3)} - P_2^{(2)} \end{bmatrix}, \qquad AX = 0$$

where $P^{(i)}$ denotes row $i$ of $P$. The null space of $A$ (smallest-singular-value right singular vector) gives $X$.

In [ ]:
def triangulate_dlt(P1, P2, x1, x2):
    points = []
    for (u1, v1), (u2, v2) in zip(x1, x2):
        A = np.array([
            u1 * P1[2] - P1[0],
            v1 * P1[2] - P1[1],
            u2 * P2[2] - P2[0],
            v2 * P2[2] - P2[1],
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        points.append(X[:3] / X[3])
    return np.array(points)

# sanity check: triangulating with the TRUE camera matrices should exactly recover the ground truth
recon_true_cameras = triangulate_dlt(P1, P2_true, x1, x2)
print(f'max error using true cameras: {np.abs(recon_true_cameras - points_3d_true).max():.2e}')

cv_points_4d = cv2.triangulatePoints(P1, P2_true, x1.T, x2.T)
cv_points_3d = (cv_points_4d[:3] / cv_points_4d[3]).T
print(f'max diff vs. cv2.triangulatePoints: {np.abs(recon_true_cameras - cv_points_3d).max():.2e}')

## Putting it together: reconstruction using *estimated* pose

Now the real test: triangulate using the camera matrix built from `recoverPose`'s *estimated* $(R, t)$, not the ground truth. Because $t$ was only recovered up to scale, so is the reconstruction &mdash; every 3D point comes out a fixed factor smaller than reality. Multiplying by the true baseline length (the one piece of information triangulation from image correspondences alone can never supply) should recover the correct metric scene.

In [ ]:
P2_estimated = K @ np.hstack([R_estimated, t_estimated.reshape(3, 1)])
reconstruction_unit_scale = triangulate_dlt(P1, P2_estimated, x1, x2)

true_baseline = np.linalg.norm(t_true)
reconstruction_metric = reconstruction_unit_scale * true_baseline

error = np.linalg.norm(reconstruction_metric - points_3d_true, axis=1)
print(f'mean 3D reconstruction error (after external scale correction): {error.mean():.2e}')
print(f'max 3D reconstruction error:                                   {error.max():.2e}')

With clean correspondences, the entire pipeline &mdash; essential matrix, pose, triangulation &mdash; reconstructs the scene to essentially floating-point precision, using nothing but 2D pixel correspondences and known intrinsics, plus *one* external number (the true baseline) to fix the scale ambiguity. In practice, that scale reference might come from a known object size in the scene, a second sensor (GPS, IMU, LiDAR), or a calibrated stereo rig (Lesson 20) instead of two arbitrary independent cameras.

### Visualizing the reconstruction

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*points_3d_true.T, c='gray', s=25, label='ground truth', alpha=0.6)
ax.scatter(*reconstruction_metric.T, c='red', s=10, marker='x', label='reconstructed')

# mark the two camera centers
cam1_center = -R1.T @ t1
cam2_center = -R_true.T @ t_true
ax.scatter(*cam1_center, c='blue', s=80, marker='^', label='camera 1')
ax.scatter(*cam2_center, c='green', s=80, marker='^', label='camera 2')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.legend(fontsize=8)
ax.set_title('Structure from motion: recovered 3D points and camera poses')
plt.show()

### Exercise

1. Add pixel noise (e.g. `rng.normal(0, 1.0, x1.shape)`) to both `x1` and `x2` before running the pipeline. How much does the reconstruction error grow, and does it grow uniformly, or worse for points farther from the cameras (think back to the disparity-depth relationship in Lesson 20 &mdash; distant points produce smaller, noisier parallax)?
2. This lesson only used two views. Real structure-from-motion pipelines (e.g. COLMAP) add many more views incrementally: each new image is *resected* against the already-reconstructed 3D points (a problem called Perspective-n-Point, or PnP &mdash; given 2D-3D correspondences and $K$, solve for that camera's pose) via `cv2.solvePnP`, then its new points are triangulated against the growing reconstruction. Look up `cv2.solvePnP`'s signature and sketch (in words) how you'd extend this notebook to a third camera.
3. Try moving `t_true` to have a much smaller magnitude (e.g. `[0.05, 0.0, 0.01]`, a very short baseline). Does pose recovery and triangulation degrade? This models a real, practical problem in SfM and stereo alike: a very small baseline gives very little parallax to work with, and even tiny pixel noise translates into large 3D uncertainty.